In [1]:
# Cell 1: GPU Check and Install Dependencies
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

# Install YOLOv26 dependencies (same as your reference notebook)
!pip install torch torchvision torchaudio
!pip install -U ultralytics supervision

CUDA available: True
GPU: Tesla T4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 25.3 MB/s eta 0:00:00


In [2]:
# Cell 2: Upgrade kagglehub
!pip install --upgrade kagglehub

In [3]:
# Cell 3: Install Kaggle API, authenticate, and download the dress dataset
!pip install kaggle
from google.colab import files
files.upload()  # Upload your kaggle.json

import os
os.environ['KAGGLE_CONFIG_DIR'] = "/content"
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

# NEW DATASET (COCO-style dress detection)
!kaggle datasets download -d sikdermdsaiful/dress-detection-using-coco-style-dataset
!unzip dress-detection-using-coco-style-dataset.zip -d /content/dataset
print("Dataset downloaded and extracted!")

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/sikdermdsaiful/dress-detection-using-coco-style-dataset
License(s): unknown
100% 33.8M/33.8M [00:00<00:00, 91.4MB/s]

Archive:  dress-detection-using-coco-style-dataset.zip
  inflating: /content/dataset/dress_detect/test/_annotations.coco.json  
  inflating: /content/dataset/dress_detect/test/images/MEN-Shirts_Polos-id_00000846-01_2_side_jpg.rf.103b1a2d9cf697196dccc452222d57bb.jpg  
  inflating: /content/dataset/dress_detect/test/images/MEN-Shirts_Polos-id_00003009-09_2_side_jpg.rf.b0aa2ce37a837f2e0b7a22cabcbb3410.jpg  
  inflating: /content/dataset/dress_detect/test/images/MEN-Sweatshirts_Hoodies-id_00006558-01_3_back_jpg.rf.46b813ebd1d31d3fc84da943bdec2ff0.jpg  
  inflating: /content/dataset/dress_detect/test/images/MEN-Tees_Tanks-id_00002103-02_3_back_jpg.rf.25a0a0f32df5d581e980e67155c7fd1a.jpg  
  inflating: /content/dataset/dress_detect/test/images/MEN-Tees_Tanks-id_00002848-02_4_full_jpg.rf.4d5a0300af7

In [4]:
# Cell 4: Explore the dataset structure (run this to see exact paths)
!ls /content/dataset
!find /content/dataset -type f | head -30

dress_detect
/content/dataset/dress_detect/test/_annotations.coco.json
/content/dataset/dress_detect/test/images/WOMEN-Dresses-id_00000224-06_1_front_jpg.rf.a37d5c0c6951dba36512f799ad08754a.jpg
/content/dataset/dress_detect/test/images/WOMEN-Pants-id_00001372-03_1_front_jpg.rf.c1c1337f918889da303dd546738e3029.jpg
/content/dataset/dress_detect/test/images/MEN-Tees_Tanks-id_00004296-01_2_side_jpg.rf.0c5f5b63939634129c5858d074c92c05.jpg
/content/dataset/dress_detect/test/images/MEN-Tees_Tanks-id_00005160-09_2_side_jpg.rf.2f7765c42a85668646613e56e4c5e6d0.jpg
/content/dataset/dress_detect/test/images/WOMEN-Dresses-id_00000416-07_3_back_jpg.rf.a478bf8784f7d7db727c0e7dbe90c9e8.jpg
/content/dataset/dress_detect/test/images/MEN-Tees_Tanks-id_00006357-01_4_full_jpg.rf.75ac543dc69c64845bbb91ccb3391c67.jpg
/content/dataset/dress_detect/test/images/WOMEN-Graphic_Tees-id_00003021-02_7_additional_jpg.rf.839dab91b6c78c25a7c1aeafb92584b0.jpg
/content/dataset/dress_detect/test/images/WOMEN-Blouses_Shirt

In [5]:
# Cell 5: Convert COCO → YOLO format (1 class: dress)
# Adjust the json_path and images_dir if your structure is different (check Cell 4)
import json
from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split

dataset_root = Path("/content/dataset")
# Common paths – change if needed after checking Cell 4
coco_json = next(dataset_root.rglob("*.json"))  # finds any .json
images_dir = next((dataset_root / "images").rglob("*.jpg"), None)  # fallback
if not images_dir:
    images_dir = dataset_root  # many datasets put images directly here

yolo_root = Path("/content/yolo_dataset")
(yolo_root / "train/images").mkdir(parents=True, exist_ok=True)
(yolo_root / "train/labels").mkdir(parents=True, exist_ok=True)
(yolo_root / "val/images").mkdir(parents=True, exist_ok=True)
(yolo_root / "val/labels").mkdir(parents=True, exist_ok=True)

with open(coco_json) as f:
    coco = json.load(f)

# Assume single class "dress" → class id 0
cls_id = 0
print(f"Classes found: {coco.get('categories', 'N/A')}")

# Group annotations by image
img_id_to_ann = {}
for ann in coco["annotations"]:
    img_id_to_ann.setdefault(ann["image_id"], []).append(ann)

images = coco["images"]
img_ids = [img["id"] for img in images]
train_ids, val_ids = train_test_split(img_ids, test_size=0.2, random_state=42)

def process_split(split_ids, split_name):
    for img_id in split_ids:
        img_info = next(i for i in images if i["id"] == img_id)
        img_name = img_info["file_name"]
        src = dataset_root / img_name
        if not src.exists():
            src = list(dataset_root.rglob(img_name))[0]  # fallback search
        dst_img = yolo_root / split_name / "images" / img_name
        shutil.copy(src, dst_img)

        label_path = yolo_root / split_name / "labels" / (Path(img_name).stem + ".txt")
        if img_id in img_id_to_ann:
            with open(label_path, "w") as f:
                for ann in img_id_to_ann[img_id]:
                    x, y, w, h = ann["bbox"]
                    x_center = (x + w/2) / img_info["width"]
                    y_center = (y + h/2) / img_info["height"]
                    width = w / img_info["width"]
                    height = h / img_info["height"]
                    f.write(f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

process_split(train_ids, "train")
process_split(val_ids, "val")
print("COCO → YOLO conversion completed!")

Classes found: [{'id': 0, 'name': 'dress', 'supercategory': 'none'}, {'id': 1, 'name': 'dress', 'supercategory': 'dress'}]
COCO → YOLO conversion completed!


In [6]:
# Cell 6: Create data.yaml for YOLOv26
data_yaml = """path: /content/yolo_dataset
train: train/images
val: val/images
nc: 1
names: ['dress']
"""

with open("/content/data.yaml", "w") as f:
    f.write(data_yaml)
print("data.yaml created for clothes detection!")

data.yaml created for clothes detection!


In [7]:
# Cell 7: Train YOLOv26n (exactly as in your reference notebook style)
from ultralytics import YOLO

# YOLOv26 ONLY
model = YOLO("yolo26n.pt")  # pretrained (recommended)

results = model.train(
    data="/content/data.yaml",
    epochs=50,          # small dataset → start with 50, increase if needed
    imgsz=640,
    batch=16,
    name="yolo26n_clothes_detection_for_human_reid",
    project="/content/runs/detect",
    exist_ok=True,
    device=0
)

print("YOLOv26n training completed!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0

In [8]:
# Cell 8: Zip and download results (same as your reference notebook)
import shutil
output_folder = "/content/runs/detect/yolo26n_clothes_detection_for_human_reid"
zip_filename = "/content/yolov26n-clothes_detection.zip"
shutil.make_archive(zip_filename.replace(".zip", ""), "zip", output_folder)
print(f"Results zipped as {zip_filename}")

Results zipped as /content/yolov26n-clothes_detection.zip


In [9]:
# Cell 9: Download the zip
from google.colab import files
files.download(zip_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>